In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import Literal,TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
import operator
load_dotenv()

In [ ]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

model2 = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

class evaluation_schema(BaseModel):
    feedback: str = Field(description="feedback of the post.")
    status: Literal['approved','rejected'] = Field("is the current post approved or rejected.")

evaluator_model = model2.with_structured_output(evaluation_schema)

In [ ]:
class PostState(TypedDict):
    topic: str
    post_content: str
    iteration: int
    max_iteration: int
    status: Literal['approved','rejected']
    feedback: str

    all_posts:Annotated[list[str],operator.add]
    all_feedbacks:Annotated[list[str],operator.add]

In [ ]:
def generate_post(state:PostState):
    prompt = f"""You are an expert LinkedIn content writer.

    Your task is to create an engaging LinkedIn post on this topic:{state['topic']}

    Requirements:
    - Write naturally, as if written by an experienced professional.
    - Hook the reader within the first two lines.
    - Keep the post between 150 and 300 words.
    - Focus on one clear message.
    - Include useful insights, lessons, or actionable takeaways.
    - Avoid generic motivational statements.
    - Avoid hashtags unless explicitly requested.
    - Do not use emojis unless requested.
    - End with a thoughtful question or call to discussion.
    - Do not explain your writing process.
    - Output only the final LinkedIn post.

    The post should be informative, engaging, authentic, and easy to read."""
    
    post = model.invoke(prompt).content
    return {"post_content":post,"all_posts":[post]}

def evaluate_post(state:PostState):
    prompt = f"""
    You are a senior LinkedIn editor.

    Your job is NOT to rewrite the post.

    Your responsibilities are:
    1. Evaluate the LinkedIn post.
    2. Decide whether it is ready for publishing.
    3. If it is not ready, provide clear and actionable feedback.

    Evaluation Criteria:

    - Strong opening hook
    - Logical flow
    - Clarity
    - Value provided to readers
    - Authentic tone
    - Reader engagement
    - Grammar and readability
    - Suitable length
    - Clear ending
    - No repetition
    - No unnecessary fluff

    Approve the post ONLY if it meets a high professional standard.

    If only tiny cosmetic improvements exist, approve it.

    If important improvements are needed, reject it.

    Rules:
    - Do not rewrite the post.
    - Do not suggest a completely different topic.
    - Feedback must be specific and actionable.
    - Maximum 6 feedback items.
    - Keep feedback concise.

    original post: {state['post_content']}
    """
    res = evaluator_model.invoke(prompt)
    return {"status":res.status, "feedback":res.feedback,"all_feedbacks":[res.feedback]}

def optimize_post(state:PostState):
    prompt = f"""
    You are an expert LinkedIn editor.

    Your task is to improve an existing LinkedIn post using reviewer feedback.

    You must:
    - Preserve the original message.
    - Preserve the author's intent.
    - Apply every relevant feedback item.
    - Improve clarity, engagement, and readability.
    - Avoid making unnecessary changes.
    - Do not ignore reviewer comments.
    - Keep the post between 150 and 300 words.
    - Return only the improved LinkedIn post.
    - Do not explain your edits.

    post: {state['post_content']}
    feedback for improvement: {state['feedback']}
    """
    post = model.invoke(prompt).content
    return {"post_content":post,"iteration":(state['iteration']+1),"all_posts":[post]}

def check_is_approved(state:PostState):
    if (state["status"] == "rejected" )or (state["iteration"] > state['max_iteration']):
        return "rejected"
    else:
        return "approved"


In [ ]:
graph = StateGraph(PostState)

graph.add_node("generate_post",generate_post)
graph.add_node("evaluate_post",evaluate_post)
graph.add_node("optimize_post",optimize_post)

graph.add_edge(START,"generate_post")
graph.add_edge("generate_post","evaluate_post")
graph.add_conditional_edges("evaluate_post",check_is_approved,{"approved":END,"rejected":"optimize_post"})
graph.add_edge("optimize_post","evaluate_post")

workflow = graph.compile()

In [ ]:
workflow.invoke({"topic":"AI in education","iteration":0,"max_iteration":5})